# ============================================
# 09_pjm_hourly_energy_downloader.ipynb
# PJM Hourly Energy Consumption Dataset
# ============================================

## Dataset Title
PJM Hourly Energy Consumption Dataset

## Source URL
Public dataset description:
https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption

Public GitHub-accessible CSV used in this notebook:
https://raw.githubusercontent.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts/master/AEP_hourly.csv

Reference repository page:
https://github.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts

## Purpose of This Dataset
This dataset is used as the real hourly energy-demand dataset for the PhD workflow.

It is especially useful for:
- hourly load forecasting
- baseline time-series modeling
- optimization input generation
- demand-profile analysis
- robustness and control experiments

## Why We Need This Dataset
We need this dataset because it provides real hourly energy-demand data from the PJM ecosystem in a simple, forecasting-ready format.

Public descriptions of the dataset state that:
- it contains over 10 years of hourly energy consumption data
- the data come from PJM
- values are reported in megawatts (MW)
- region-specific files are used

This makes it highly suitable for:
- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 11: Full System Simulation

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download a real PJM-region CSV from a public GitHub raw URL
4. Validate the file
5. Save a dataset manifest
6. Create a clean ZIP bundle
7. Save everything in Google Drive
8. Create a final package ZIP
9. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 11: Full System Simulation

## Notes
Public descriptions state that the PJM Hourly Energy Consumption dataset contains over 10 years of hourly energy consumption data from PJM in megawatts, with region-level files such as AEP, COMED, DAYTON, and PJME.

In [1]:
# ============================================
# 09_pjm_hourly_energy_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import json
import shutil
import zipfile
from pathlib import Path
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "PJM Hourly Energy Consumption Dataset"
SOURCE_PAGE = "https://github.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts"
DOWNLOAD_URL = "https://raw.githubusercontent.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts/master/AEP_hourly.csv"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "09_pjm_hourly_energy"
raw_dir = dataset_root / "raw"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=300, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download CSV
# -----------------------------
csv_path = raw_dir / "AEP_hourly.csv"

if not csv_path.exists():
    print("\nDownloading dataset CSV...")
    download_file(DOWNLOAD_URL, csv_path)
    print("Download complete:", csv_path)
else:
    print("\nUsing cached file:", csv_path)

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

print("CSV size:", round(csv_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 6. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(csv_path),
    "used_in_next_notebooks": [
        "Notebook 2: Baseline Models",
        "Notebook 3: NEO-Forecast",
        "Notebook 6: NEO-Opt",
        "Notebook 7: NEO-Robust",
        "Notebook 11: Full System Simulation"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 7. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "pjm_hourly_energy_CLEAN_bundle.zip"

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname="raw/" + csv_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 8. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "pjm_hourly_energy_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname="raw/" + csv_path.name)
    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 9. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/pjm_hourly_energy_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 10. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw CSV              :", csv_path)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 11. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : PJM Hourly Energy Consumption Dataset
Source Page  : https://github.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts
Download URL : https://raw.githubusercontent.com/soumilshah1995/Data-Analysis-Over-10-years-of-hourly-energy-consumption-data-from-PJM-in-Megawatts/master/AEP_hourly.csv
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy
 - /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy/bundle

Download complete: /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy/raw/AEP_hourly.csv
CSV size: 3.24 MB

Saved manifest: /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy/dataset_manifest.json
Created clean bundle ZIP: /content/drive/MyDrive/PhD_Thesis_Datasets/09_pjm_hourly_energy/bundle/pjm_hourly_energy_CLE

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>